# Applying SMOTE

Comparing logistic regression on the normal data vs data upsampled with SMOTE. Each genre combo (like Action|Adventure|Fantasy) is treated as its own class.

In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score

In [ ]:
# import data
anime = pd.read_csv('cleaned_animes.csv')
anime = anime.dropna(subset = ['synopsis', 'genres'])

# combine title and synopsis like in classifying.ipynb
anime['title_english'] = anime['title_english'].fillna('')
anime['text'] = anime['title_english'] + ' ' + anime['synopsis']

In [ ]:
# get rid of genre combos that don't have enough samples to upsample
# (we picked 30 as the cutoff)
counts = anime['genres'].value_counts()
keep = counts[counts >= 30].index
df = anime[anime['genres'].isin(keep)]

print(len(keep), "genre combos kept")
print(len(df), "rows kept")

In [ ]:
# train/test split
# stratify makes sure every genre combo is in both train and test
X_train_text, X_test_text, y_train, y_test = train_test_split(df['text'], df['genres'], test_size = 0.2, random_state = 42, stratify = df['genres'])

In [ ]:
# TF-IDF vectorization (same as classifying.ipynb)
vectorizer = TfidfVectorizer(max_features = 5000, stop_words = 'english', ngram_range = (1, 2))
X_train = vectorizer.fit_transform(X_train_text) # fit only on training data
X_test = vectorizer.transform(X_test_text)

In [ ]:
# SMOTE
# 'not majority' upsamples every class to be the same size as the biggest class
# only do this on the training data, test data stays the same
smote = SMOTE(sampling_strategy = 'not majority', k_neighbors = 5, random_state = 42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("before smote:", X_train.shape[0], "rows")
print("after smote:", X_train_smote.shape[0], "rows")
print(pd.Series(y_train_smote).value_counts()) # all the same size now

In [ ]:
# training models
model_normal = LogisticRegression(max_iter = 5000, random_state = 42)
model_normal.fit(X_train, y_train)

model_smote = LogisticRegression(max_iter = 5000, random_state = 42)
model_smote.fit(X_train_smote, y_train_smote)

# making predictions on the same test set
y_pred_normal = model_normal.predict(X_test)
y_pred_smote = model_smote.predict(X_test)

In [ ]:
# evaluate!!
print("normal data")
print("accuracy:", round(accuracy_score(y_test, y_pred_normal), 4))
print("f1:", round(f1_score(y_test, y_pred_normal, average = 'weighted'), 4))

print()
print("smote data")
print("accuracy:", round(accuracy_score(y_test, y_pred_smote), 4))
print("f1:", round(f1_score(y_test, y_pred_smote, average = 'weighted'), 4))

In [ ]:
# f1 for each genre combo
print("NORMAL DATA")
print(classification_report(y_test, y_pred_normal, zero_division = 0))

print("SMOTE DATA")
print(classification_report(y_test, y_pred_smote, zero_division = 0))